# nb_monitoramento
Verifica a saúde do DW e grava os problemas em `ctl_alertas` (lakehouse Bronze):
1. **Rotina atrasada**: rotina do nb_rotina_nexti sem executar há mais que o limite, dentro da janela;
2. **Tabela parada**: tabela incremental do ciclo geral com o watermark sem avançar além do limite;
3. **Falhas**: etapas com FALHA/PARCIAL no `ctl_log` desde a última verificação;
4. **Volume**: tabela FULL do ciclo completo com queda grande em relação à média dos últimos ciclos;
5. **Qualidade**: regras do `ctl_qualidade` quebradas na última execução.

Limites em `config/qualidade.yml` (seção `monitoramento`). O mesmo alerta não se repete dentro de
`silenciar_horas`. Com o segredo do webhook no Key Vault, os alertas de severidade erro vão para o Teams.

In [ ]:
configuracao_api = '{"NEXTI": {"key_vault": "https://kv-dw-fabric-kmf.vault.azure.net/"}}'
horas_log = 24          # janela do ctl_log analisada na primeira execução

In [ ]:
import os, json, datetime as dt
import pyarrow as pa
import duckdb
from deltalake import DeltaTable, write_deltalake

# Não renomear os lakehouses: o banco do SQL endpoint guarda o nome da criação e a sincronização falha
LH_BRONZE, LH_SILVER, LH_GOLD = "Bronze", "Silver", "Gold"
FUSO_BRT = dt.timezone(dt.timedelta(hours=-3))

_LOCAL = os.environ.get("DW_LOCAL_ROOT")  # usado só em testes fora do Fabric
if _LOCAL:
    def _opts():
        return None
    def caminho(lakehouse, tabela):
        return f"{_LOCAL}/{lakehouse}/{tabela}"
    def sair(valor):
        globals()["_EXIT_VALUE"] = valor
else:
    import notebookutils
    _WS = notebookutils.runtime.context["currentWorkspaceName"]
    def _opts():
        return {"bearer_token": notebookutils.credentials.getToken("storage"),
                "use_fabric_endpoint": "true"}
    def caminho(lakehouse, tabela):
        return f"abfss://{_WS}@onelake.dfs.fabric.microsoft.com/{lakehouse}.Lakehouse/Tables/{tabela}"
    def sair(valor):
        notebookutils.notebook.exit(valor)

def existe(lakehouse, tabela):
    try:
        DeltaTable(caminho(lakehouse, tabela), storage_options=_opts())
        return True
    except Exception:
        return False

def ler(lakehouse, tabela, filtros=None, colunas=None):
    dt_ = DeltaTable(caminho(lakehouse, tabela), storage_options=_opts())
    kw = {k: v for k, v in (("filters", filtros), ("columns", colunas)) if v}
    return dt_.to_pyarrow_table(**kw)

def gravar(lakehouse, tabela, dados, modo="append"):
    if modo == "append":
        acrescentar_colunas(lakehouse, tabela, dados.schema)
    kw = {"schema_mode": "overwrite"} if modo == "overwrite" else {}
    write_deltalake(caminho(lakehouse, tabela), dados, mode=modo,
                    storage_options=_opts(), **kw)

def acrescentar_colunas(lakehouse, tabela, schema):
    """Coluna nova na origem (ex.: campo novo da API): a tabela é regravada com a coluna, vazia nas linhas
    antigas, antes do append ou do MERGE. Não depende da versão do deltalake suportar evolução de schema."""
    if not existe(lakehouse, tabela):
        return
    atual = tabela_delta(lakehouse, tabela)
    nomes = {f.name for f in atual.schema().fields}
    novas = [c for c in schema if c.name not in nomes]
    if not novas:
        return
    dados = atual.to_pyarrow_table()
    for c in novas:
        dados = dados.append_column(c, pa.nulls(dados.num_rows, type=c.type))
    write_deltalake(caminho(lakehouse, tabela), dados, mode="overwrite", schema_mode="overwrite",
                    storage_options=_opts())
    print(f"{tabela}: colunas novas {[c.name for c in novas]}")

def tabela_delta(lakehouse, tabela):
    return DeltaTable(caminho(lakehouse, tabela), storage_options=_opts())

def para_arrow(rel):
    """Converte resultado DuckDB em pyarrow.Table (compatível com várias versões)."""
    r = rel.fetch_arrow_table() if hasattr(rel, "fetch_arrow_table") else rel.arrow()
    return r.read_all() if hasattr(r, "read_all") else r

def nome_bronze(tabela_destino):
    """ctl_tabelas guarda o nome base (fato_contas_receber_gs); cada camada recebe o seu prefixo."""
    return f"bronze_{tabela_destino}"

def nome_silver(tabela_destino):
    return f"silver_{tabela_destino}"

def agora_utc():
    return dt.datetime.now(dt.timezone.utc)

def watermark_gold(gold, fonte):
    """Último _dt_carga da Silver já levado para esta Gold (tabela ctl_gold na Bronze). As Golds são
    incrementais por esta marca, e não pelo lote: se uma Gold falhar, a próxima execução retoma dali."""
    if not existe(LH_BRONZE, "ctl_gold"):
        return None
    for r in ler(LH_BRONZE, "ctl_gold").to_pylist():
        if r["gold"] == gold and r["fonte"] == fonte:
            return r["ultimo_dt_carga"]
    return None

def gravar_watermark_gold(gold, fonte, valor):
    linha = pa.Table.from_pylist([{"gold": gold, "fonte": fonte, "ultimo_dt_carga": valor,
                                   "dt_atualizacao": agora_utc()}],
                                 schema=pa.schema([("gold", pa.string()), ("fonte", pa.string()),
                                                   ("ultimo_dt_carga", pa.timestamp("us", tz="UTC")),
                                                   ("dt_atualizacao", pa.timestamp("us", tz="UTC"))]))
    if not existe(LH_BRONZE, "ctl_gold"):
        gravar(LH_BRONZE, "ctl_gold", linha, modo="overwrite")
        return
    (tabela_delta(LH_BRONZE, "ctl_gold")
        .merge(linha, predicate="t.gold = s.gold AND t.fonte = s.fonte", source_alias="s", target_alias="t")
        .when_matched_update_all().when_not_matched_insert_all().execute())

def consulta(nome):
    """Texto de uma consulta da pasta `consultas` (no Fabric: lakehouse Gold → Files/consultas/<nome>.sql,
    copiada de implantacao/consultas pela implantação). Nos testes locais, lê de DW_CONSULTAS."""
    if _LOCAL:
        with open(os.path.join(os.environ["DW_CONSULTAS"], f"{nome}.sql"), encoding="utf-8") as arq:
            return arq.read()
    return notebookutils.fs.head(
        f"abfss://{_WS}@onelake.dfs.fabric.microsoft.com/{LH_GOLD}.Lakehouse/Files/consultas/{nome}.sql", 10_000_000)

def cabecalho(sql):
    """Configuração no topo do .sql, em linhas `-- @chave valor` (ex.: `-- @destino gold_fato_coberturas`)."""
    cfg = {}
    for linha in sql.splitlines():
        linha = linha.strip()
        if linha.startswith("-- @"):
            chave, _, valor = linha[4:].partition(" ")
            cfg[chave] = valor.strip()
    return cfg

def ler_config(nome):
    """Arquivo YAML da pasta config (no Fabric: lakehouse Gold → Files/config/<nome>, copiado de config/ pela
    implantação). Nos testes locais, lê de DW_CONFIG."""
    import yaml
    if _LOCAL:
        with open(os.path.join(os.environ["DW_CONFIG"], nome), encoding="utf-8") as arq:
            return yaml.safe_load(arq)
    return yaml.safe_load(notebookutils.fs.head(
        f"abfss://{_WS}@onelake.dfs.fabric.microsoft.com/{LH_GOLD}.Lakehouse/Files/config/{nome}", 10_000_000))

In [ ]:
cfg = ler_config("qualidade.yml")["monitoramento"]
agora = agora_utc()
agora_brt = dt.datetime.now(FUSO_BRT)
alertas = []

def alerta(tipo, objeto, severidade, mensagem):
    alertas.append({"dt_alerta": agora, "tipo": tipo, "objeto": objeto, "severidade": severidade,
                    "mensagem": mensagem[:4000], "chave": f"{tipo}|{objeto}"})

# ---------------------------------------------------------------- 1. rotinas atrasadas
for rotina, lim in cfg.get("rotinas", {}).items():
    ini, fim = lim["janela"]
    if not (ini <= agora_brt.strftime("%H:%M") <= fim):
        continue
    ultima = watermark_gold(f"ROTINA_{rotina}", "execucao")
    # tolerância no começo da janela: a primeira execução do dia ainda pode não ter rodado
    inicio_janela = agora_brt.replace(hour=int(ini[:2]), minute=int(ini[3:]), second=0, microsecond=0)
    referencia = max(ultima, inicio_janela) if ultima else inicio_janela
    atraso = (agora - referencia).total_seconds() / 60
    if atraso > lim["atraso_max_min"]:
        desde = f"desde {ultima.astimezone(FUSO_BRT):%d/%m %H:%M}" if ultima else "nunca executou"
        alerta("ROTINA_ATRASADA", rotina, "erro", f"Rotina {rotina} sem executar há {atraso:.0f} min ({desde})")

# ---------------------------------------------------------------- 2. tabelas incrementais paradas
limite_h = cfg.get("tabelas_incrementais_max_horas", 30)
for t in ler(LH_BRONZE, "ctl_tabelas").to_pylist():
    if not t["ativo"] or t["modo_carga"] != "INCREMENTAL" or t["dt_atualizacao"] is None:
        continue
    horas = (agora - t["dt_atualizacao"]).total_seconds() / 3600
    if horas > limite_h:
        alerta("TABELA_PARADA", t["id"], "alerta",
               f"{t['id']} sem avançar há {horas:.0f} h (último watermark {t['ultimo_watermark']})")

# ---------------------------------------------------------------- 3. falhas no log
ultima_verificacao = watermark_gold("MONITORAMENTO", "ctl_log") or (agora - dt.timedelta(hours=horas_log))
log = ler(LH_BRONZE, "ctl_log", filtros=[("dt_registro", ">", ultima_verificacao)]).to_pylist()
for x in log:
    if x["status"] in ("FALHA", "PARCIAL") and x["etapa"] in ("CICLO", "ROTINA", "GOLD", "SILVER", "BRONZE"):
        alerta(f"FALHA_{x['etapa']}", f"{x['ciclo']}|{x['tabela_id']}", "erro",
               f"{x['ciclo']} {x['tabela_id']} ({x['etapa']}) {x['status']}: {x['mensagem'][:500]}")

# ---------------------------------------------------------------- 4. volume das tabelas FULL
queda = cfg.get("volume_queda_max_percentual", 70) / 100
completos = ler(LH_BRONZE, "ctl_log", filtros=[("ciclo", "=", "COMPLETO"), ("etapa", "=", "SILVER"),
                                               ("dt_registro", ">", agora - dt.timedelta(days=15))]).to_pylist()
por_tabela = {}
for x in sorted(completos, key=lambda x: x["dt_registro"]):
    if x["status"] == "SUCESSO":
        por_tabela.setdefault(x["tabela_id"], []).append(x)
modos = {t["id"]: t["modo_carga"] for t in ler(LH_BRONZE, "ctl_tabelas").to_pylist()}
for tid, xs in por_tabela.items():
    if modos.get(tid) != "FULL" or len(xs) < 3 or xs[-1]["dt_registro"] <= ultima_verificacao:
        continue
    media = sum(x["linhas"] for x in xs[:-1]) / len(xs[:-1])
    if media > 0 and xs[-1]["linhas"] < media * (1 - queda):
        alerta("VOLUME_BAIXO", tid, "alerta",
               f"{tid}: {xs[-1]['linhas']} linhas no último ciclo completo, média de {media:.0f}")

# ---------------------------------------------------------------- 5. qualidade
if existe(LH_BRONZE, "ctl_qualidade"):
    qual = ler(LH_BRONZE, "ctl_qualidade", filtros=[("dt_execucao", ">", ultima_verificacao)]).to_pylist()
    for x in qual:
        if x["status"] != "OK":
            alerta("QUALIDADE", f"{x['grupo']}|{x['regra']}", x["severidade"],
                   f"{x['regra']} ({x['tabela']}): {x['status']}, {x['linhas_problema']} linhas. Exemplos: {x['amostra'][:300]}")

In [ ]:
# não repete o mesmo alerta dentro de silenciar_horas
silenciar = dt.timedelta(hours=cfg.get("silenciar_horas", 6))
recentes = set()
if existe(LH_BRONZE, "ctl_alertas"):
    recentes = {a["chave"] for a in ler(LH_BRONZE, "ctl_alertas", filtros=[("dt_alerta", ">", agora - silenciar)],
                                        colunas=["chave"]).to_pylist()}
novos = list({a["chave"]: a for a in alertas if a["chave"] not in recentes}.values())   # 1 por chave
schema = pa.schema([("dt_alerta", pa.timestamp("us", tz="UTC")), ("tipo", pa.string()), ("objeto", pa.string()),
                    ("severidade", pa.string()), ("mensagem", pa.string()), ("chave", pa.string())])
if novos:
    gravar(LH_BRONZE, "ctl_alertas", pa.Table.from_pylist(novos, schema=schema),
           modo="append" if existe(LH_BRONZE, "ctl_alertas") else "overwrite")
gravar_watermark_gold("MONITORAMENTO", "ctl_log", agora)

# Teams: só os novos de severidade erro, numa mensagem só
erros = [a for a in novos if a["severidade"] == "erro"]
if erros and not _LOCAL and cfg.get("teams_webhook_segredo"):
    try:
        cofre = json.loads(configuracao_api)["NEXTI"]["key_vault"]
        url = notebookutils.credentials.getSecret(cofre, cfg["teams_webhook_segredo"])
        import requests
        texto = "**DW Fabric: alertas**\n\n" + "\n\n".join(f"- {a['mensagem']}" for a in erros[:20])
        requests.post(url, json={"text": texto}, timeout=30)
    except Exception as e:   # sem webhook configurado ou fora do ar: o alerta continua no ctl_alertas
        print(f"Teams não notificado: {str(e)[:200]}")

for a in novos:
    print(f"{a['severidade']:<7} {a['tipo']:<16} {a['mensagem']}")
sair(json.dumps({"alertas": len(alertas), "novos": len(novos), "erros": len(erros)}))